In [1]:
# Imports
%matplotlib inline
%load_ext autoreload
%autoreload 2

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import torch
import os
from sklearn.linear_model import LogisticRegression


In [12]:
from src.config_presets.tools.get_config import get_config
from src.constants import PATIENT_ID_COL_NAME, PATIENT_ID_LENGTHS_DICT, EARLY_NTCP_TIMEPOINTS, LATE_NTCP_TIMEPOINTS

config = get_config('Base_config')

config['data']['source'] = "MDACC"

PATIENT_ID_LENGTH = PATIENT_ID_LENGTHS_DICT[config['data']['source']]


src\config_presets\Base_config.yaml
src\config_presets\Base_config.yaml


In [18]:
df_features_dir = r"\\zkh\appdata\RTDicom\Projectline_HNC_modelling\Users\Daniel MacRae\1. MultiTox_HNC\DL_NTCP_Multitox\datasets\MT_dataset\stratified_sampling_test_542.csv"

#df_features_dir = config['paths']['csv']

df_features = pd.read_csv(df_features_dir, delimiter=';')
df_features[PATIENT_ID_COL_NAME] = df_features[PATIENT_ID_COL_NAME].astype(str).str.zfill(PATIENT_ID_LENGTH)

df_features['Leeftijd'] = df_features['Age']
df_features['Parotid_sum_of_sqrts'] = df_features[['Parotid_L_meandose',
 'Parotid_R_meandose']].apply(lambda x: np.sqrt(x[0]) + np.sqrt(x[1]), axis=1).astype(float)
df_features["Parotid_sum_sqrt"] = df_features['Parotid_sum_of_sqrts']

df_features['Parotid_log_of_mean'] = np.log(df_features['Parotid_meandose'])
df_features['Oral_Cavity_sqrt_of_mean'] = np.sqrt(df_features['OralCavity_Ext_meandose'])
df_features


MDACC = True

if not MDACC:
    df_second_validation_set = pd.read_csv(r"\\zkh\appdata\RTDicom\Projectline_HNC_modelling\Users\Daniel MacRae\1. MultiTox_HNC\Dataset\MT_post_2021_validation_set.csv", delimiter=';')
    df_second_validation_set['Leeftijd'] = df_second_validation_set['Age']

if MDACC:
    df_second_validation_set = pd.read_csv(r"\\zkh\appdata\RTDicom\Projectline_HNC_modelling\Users\Daniel MacRae\1. MultiTox_HNC\DL_NTCP_Multitox\datasets\MDACC_dataset\MDACC_stratified_sampling_test_CITOR.csv", delimiter=';')
    df_second_validation_set['Leeftijd'] = df_second_validation_set['Age']

    df_second_validation_set['Leeftijd'] = df_second_validation_set['Age']
    df_second_validation_set['Parotid_sum_of_sqrts'] = df_second_validation_set[['Parotid_L_meandose',
    'Parotid_R_meandose']].apply(lambda x: np.sqrt(x[0]) + np.sqrt(x[1]), axis=1).astype(float)

    df_second_validation_set['Parotid_log_of_mean'] = np.log(df_second_validation_set['Parotid_meandose'])
    df_second_validation_set['Oral_Cavity_sqrt_of_mean'] = np.sqrt(df_second_validation_set['OralCavity_Ext_meandose'])
    df_second_validation_set["Parotid_sum_sqrt"] = df_second_validation_set['Parotid_sum_of_sqrts']

    #config['data']['columns'] = 
    config['columns']['clinical_features'] = [col_name.replace("W01", "BSL") for col_name in config['columns']['clinical_features']]



C:\Users\macraedc\AppData\Local\Temp\ipykernel_22732\1295194635.py:10: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  'Parotid_R_meandose']].apply(lambda x: np.sqrt(x[0]) + np.sqrt(x[1]), axis=1).astype(float)
C:\Users\macraedc\AppData\Local\Temp\ipykernel_22732\1295194635.py:30: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  'Parotid_R_meandose']].apply(lambda x: np.sqrt(x[0]) + np.sqrt(x[1]), axis=1).astype(float)


In [19]:
DL_trial_dir = r"\\zkh\appdata\RTDicom\Projectline_HNC_modelling\Users\Daniel MacRae\1. MultiTox_HNC\Feb_2025_results\Trial_32"

# get the folders in the directory (i.e. the folder per trial)
folder_names = os.listdir(DL_trial_dir)
folder_names = [folder for folder in folder_names if os.path.isdir(os.path.join(DL_trial_dir, folder))]


for k_fold_idx, folder_name in enumerate(folder_names):
    k_fold_folder_dir = os.path.join(DL_trial_dir, folder_name)
    config['general']['resultsCurrentDirectory'] = k_fold_folder_dir
    
    print(k_fold_folder_dir)
    model_preds_df = pd.read_csv(os.path.join(k_fold_folder_dir, "model_predictions.csv"), delimiter=";")

    # check if this df contains the test predictions as well, if not, then also load in the test_set_predictions.csv
    if "test" not in model_preds_df["Mode"].values:
        test_preds_df = pd.read_csv(os.path.join(k_fold_folder_dir, "test_set_predictions.csv"), delimiter=";")
        model_preds_df = pd.concat([model_preds_df, test_preds_df], axis=0)

    model_preds_df[PATIENT_ID_COL_NAME] = model_preds_df[PATIENT_ID_COL_NAME].astype(str).str.zfill(PATIENT_ID_LENGTH)

    train_patient_IDs = model_preds_df[model_preds_df["Mode"] == "train"][PATIENT_ID_COL_NAME].values
    val_patient_IDs = model_preds_df[model_preds_df["Mode"] == "val"][PATIENT_ID_COL_NAME].values
    test_patient_IDs = model_preds_df[model_preds_df["Mode"] == "test"][PATIENT_ID_COL_NAME].values

    run_logistic_regression_posthoc(config, df_features, train_patient_IDs=train_patient_IDs, val_patient_IDs=val_patient_IDs, test_patient_IDs=test_patient_IDs,
                                    df_test=df_second_validation_set, test_source='MDACC')

    

    
    

\\zkh\appdata\RTDicom\Projectline_HNC_modelling\Users\Daniel MacRae\1. MultiTox_HNC\Feb_2025_results\Trial_32\KFold1
['intercept', 'Submandibular_meandose', 'Parotid_sum_sqrt', 'Xerostomia_W01_Heel_erg', 'Xerostomia_W01_Een_beetje'] [-2.4795669402601686, 0.02113826280461968, 0.10321768526526263, 2.8498354954886045, 0.7289954282206375]
\\zkh\appdata\RTDicom\Projectline_HNC_modelling\Users\Daniel MacRae\1. MultiTox_HNC\Feb_2025_results\Trial_32\KFold2
['intercept', 'Submandibular_meandose', 'Parotid_sum_sqrt', 'Xerostomia_W01_Heel_erg', 'Xerostomia_W01_Een_beetje'] [-2.4677705597010178, 0.020526594839806064, 0.1058359638700763, 1.1251889068619607, 0.7082230611176347]
\\zkh\appdata\RTDicom\Projectline_HNC_modelling\Users\Daniel MacRae\1. MultiTox_HNC\Feb_2025_results\Trial_32\KFold3
['intercept', 'Submandibular_meandose', 'Parotid_sum_sqrt', 'Xerostomia_W01_Heel_erg', 'Xerostomia_W01_Een_beetje'] [-2.486172995173002, 0.02112594241047404, 0.10529832833703404, 1.1628561759313656, 0.77705921

In [ ]:
df_features.Parotid

,PatientID,Sex,Age,CT+C_available,CT_Artefact,Loctum2,Photons,T_stage,N_stage,Smoking,...,Dysphagia_W01_Grade2,Dysphagia_W01_Grade3_4,Dysphagia_W01_Nogal,Dysphagia_W01_Heel_erg,Dysphagia_W01_Een_beetje,Dysphagia_W01_Helemaal_niet,Leeftijd,Parotid_sum_of_sqrts,Parotid_log_of_mean,Oral_Cavity_sqrt_of_mean
0,0000005680,1,0.68,1.0,1,Larynx,1.0,T2,N0,1,...,0,0,0,0,0,0,0.68,1.143999,-1.118512,0.745590
1,0000020715,1,0.56,1.0,1,Pharynx,1.0,T4,N1,1,...,0,0,0,0,0,0,0.56,9.353729,3.111759,7.528354
2,0000021879,1,0.67,1.0,0,Pharynx,1.0,T2,N2,1,...,0,0,0,0,0,0,0.67,11.298645,3.459127,7.053742
3,0000052277,1,0.50,1.0,0,Pharynx,1.0,T4,N2,1,...,1,0,0,0,0,0,0.50,11.448122,3.490724,7.947537
4,0000059896,1,0.51,1.0,2,Pharynx,1.0,T2,N2,1,...,0,0,0,0,0,0,0.51,11.256083,3.465100,6.692084
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1085,0009946432,1,0.75,1.0,0,Larynx,1.0,T3,N0,1,...,0,0,0,0,0,0,0.75,8.624142,2.933142,4.502776
1086,0009949620,1,0.75,0.0,1,Larynx,1.0,T1,N0,0,...,0,0,0,0,0,0,0.75,0.578479,-2.472870,0.453823
1087,0009956433,1,0.66,1.0,2,Pharynx,1.0,T2,N2,0,...,0,0,0,0,0,0,0.66,10.014044,3.242149,7.247602
1088,0009977441,1,0.50,1.0,0,Larynx,1.0,T3,N0,0,...,0,0,0,0,0,0,0.50,8.289383,2.843876,4.137662


In [10]:
def refit_CITOR_model(config, df_train, endpoint = "Xerostomia_M06", CITOR_model_name = 'xerostomia_late'):
    
    if len(config['CITOR'][CITOR_model_name]['models'].keys()) > 1:
        feature_dict = {'model1':{}, 'model2':{}}
        submodel_names = ['model1', 'model2']

        #all_feature_columns = list(set()) 
    else:
        feature_dict = {'model1':{}}
        submodel_names = ['model1']   

    
    #all_feature_columns = list()  # TODO

    # refit (all of the sub-) models
    for submodel_names in submodel_names:
        # select the columns per submodel
        feature_column_names = np.array(config['CITOR'][CITOR_model_name]['models'][submodel_names]['features'])

        X_train = df_train[feature_column_names]
        y_train = df_train[endpoint]

        # make and fit the (sub)model
        sub_model = LogisticRegression(random_state=config['general']['seed'], C=1e10, max_iter = 1000)

        # Messy things to handle any missing data
        valid_indices = y_train != -1
        X_train = X_train[valid_indices]
        y_train = y_train[valid_indices]
        
        # fit the model
        sub_model.fit(X_train, y_train)

        # save the submodel in the feature_dict
        feature_dict[submodel_names]['features'] = feature_column_names
        feature_dict[submodel_names]['coef'] = sub_model.coef_[0]
        feature_dict[submodel_names]['intercept'] = sub_model.intercept_[0]

        # print(f"submodel {submodel_names} has been fitted")
        # sub_model_dict = {f: c for f, c in zip(feature_column_names, sub_model.coef_[0])}
        # print(sub_model_dict)
        # print(feature_dict[submodel_names]['intercept'])


    # combine the submodels
    if len(feature_dict.keys()) > 1:
        model1 = list(feature_dict.keys())[0]
        model2 = list(feature_dict.keys())[1]
        features = []
        coef = []
        features.append('intercept')
        coef.append(.5*(feature_dict[model1]['intercept'] + feature_dict[model2]['intercept']))
        
        features1 = feature_dict[model1]['features']
        features2 = feature_dict[model2]['features']
        
        overlap =  list(set(features1) & set(features2))
        remainder_m1 = list(set(features1) - set(features2))
        remainder_m2 = list(set(features2) - set(features1))

        
        for i in range(len(remainder_m1)):
            features.append(remainder_m1[i])
            coef.append(.5*(feature_dict[model1]['coef'][np.where(features1 == remainder_m1[i])])[0])         
        for i in range(len(remainder_m2)):
            features.append(remainder_m2[i])
            coef.append(.5*(feature_dict[model2]['coef'][np.where(features2 == remainder_m2[i])])[0])    
        for i in range(len(overlap)):
            features.append(overlap[i])
            coef.append(.5*(feature_dict[model1]['coef'][np.where(features1 == overlap[i])]+feature_dict[model2]['coef'][np.where(features2 == overlap[i])])[0])     
        print(features, coef)
    else:
        features = ['intercept'] + list(feature_column_names)
        coef =[sub_model.intercept_[0]] + list(sub_model.coef_[0]) 

        print(features, coef)


    # print(features)
    # features_coeff_dict = {k: v for k, v in zip(features, coef)}
    # print(features_coeff_dict)
    # print(coef[1:])
    
    model = LogisticRegression(random_state=config['general']['seed'], C=1e10, max_iter = 1000, multi_class='ovr')
    model.intercept_ = np.array([coef[0]])
    model.coef_ = np.array([coef[1:]])

    

    return model, features[1:] 
    




In [5]:
from src.utils.saving.saving_predictions import concatenate_predictions, save_predictions


def determine_CITOR_model_name(endpoint_name):
    endpoint_name = endpoint_name.lower()

    toxicity_name = "_".join(endpoint_name.split("_")[:-1])
    if toxicity_name == 'sticky':
        toxicity_name = 'sticky_saliva'

    timepoint = endpoint_name.split("_")[-1]

    CITOR_name = toxicity_name
    if timepoint in EARLY_NTCP_TIMEPOINTS:
        CITOR_name += "_early"
    else:
        CITOR_name += "_late"

    return CITOR_name




from src.evaluation.total_evaluation import total_evaluation_current_fold
from src.evaluation.get_visualisations import get_visualizations


def run_logistic_regression_posthoc(config, df_features, train_patient_IDs, val_patient_IDs, test_patient_IDs=None, df_test=None, test_source=None):
    # get the train and val patients' data
    df_train = df_features[df_features[PATIENT_ID_COL_NAME].isin(train_patient_IDs)]
    df_val = df_features[df_features[PATIENT_ID_COL_NAME].isin(val_patient_IDs)]
    assert len(df_train) == len(train_patient_IDs)
    assert len(df_val) == len(val_patient_IDs)

    # get the test patients' data (if you want that)
    if df_test is None:
        if test_patient_IDs is not None:
            df_test = df_features[df_features[PATIENT_ID_COL_NAME].isin(test_patient_IDs)]
            assert len(df_test) == len(test_patient_IDs)
        else:
            df_test = None
    else:
        df_test = df_test
    # print(df_train.shape)
    # print(df_val.shape)
    # print(df_test.shape)

    # places to store the results
    train_y_pred_dict, train_y_true_dict = dict(), dict()
    val_y_pred_dict, val_y_true_dict = dict(), dict()
    test_y_pred_dict, test_y_true_dict = dict(), dict()
    
    endpoint_list = config['columns']['labels']

    all_preds_dict = dict()
    all_targets_dict = dict()
    

    for idx, endpoint in enumerate(endpoint_list):
        #print(f"fitting models for {endpoint}")

        # get the features and labels
        #X_train = df_train.loc[:, ]
        CITOR_model_name = determine_CITOR_model_name(endpoint)

        model, feature_column_names = refit_CITOR_model(config, df_train, endpoint = endpoint, CITOR_model_name = CITOR_model_name)

        #print("\n\n")
                         # endpoint, train_patient_IDs, val_patient_IDs, test_patient_IDs)

        # get the predictions
        df_X = df_features.loc[:, feature_column_names]

        df_train_X = df_train.loc[:, feature_column_names].values
        df_val_X = df_val.loc[:, feature_column_names].values

        train_patient_IDs = list(df_train[PATIENT_ID_COL_NAME].values)
        val_patient_IDs = list(df_val[PATIENT_ID_COL_NAME].values)

        train_preds = model.predict_proba(df_train_X)[:,1]
        val_preds = model.predict_proba(df_val_X)[:,1]

        if test_patient_IDs is not None:
            df_test_X = df_test.loc[:, feature_column_names].values
            test_patient_IDs = list(df_test[PATIENT_ID_COL_NAME].values)
            test_preds = model.predict_proba(df_test_X)[:,1]
            #print(df_test.shape)
            #print(test_preds.shape)
        else:
            df_test_X = None
            test_patient_IDs = []
            test_preds = None


        #preds = model.predict_proba(df_X)[:,1]

        # store the predictions
        # all_preds_dict[endpoint] = np.concatenate([train_preds, val_preds, test_preds])
        # all_targets_dict[endpoint] = np.concatenate([df_train[endpoint].values, df_val[endpoint].values, df_test[endpoint].values])
        all_preds_dict[endpoint] = np.concatenate([test_preds])
        all_targets_dict[endpoint] = np.concatenate([df_test[endpoint].values])

    # all_patientIDs_list = train_patient_IDs + val_patient_IDs + test_patient_IDs
    # mode_list = ["train"] * len(train_patient_IDs) + ["val"] * len(val_patient_IDs) + ["test"] * len(test_patient_IDs)

    all_patientIDs_list = test_patient_IDs
    mode_list = ["test"] * len(test_patient_IDs)

    if test_source is not None:
        config['data']['source'] = test_source
    # save all the predictions into one csv file
    # save_predictions(config, all_patientIDs_list, all_preds_dict, all_targets_dict, mode_list, filename=f"all_lr_predictions.csv")
    # #print("saved all predictions")

    # pred_csv_dir = os.path.join(config['general']['resultsCurrentDirectory'], f"all_lr_predictions_{test_source}.csv")
    # total_evaluation_current_fold(config, sets = ['test'], pred_csv_dir = pred_csv_dir, lr=True)

    # get_visualizations(config, sets=['test'], pred_csv_dir=pred_csv_dir, lr=True)

    #print(all_preds_dict[endpoint].shape)




    
    # get the features and labels

    
    
    



# Make ensemble preds

In [18]:
# do the ensembling of test set CITOR preds

fold_test_set_preds_list = []
test_patientIDs_list_fold1 = None


for k_fold_idx, folder_name in enumerate(folder_names):
    k_fold_folder_dir = os.path.join(DL_trial_dir, folder_name)
    config['general']['resultsCurrentDirectory'] = k_fold_folder_dir
    
    #print(k_fold_folder_dir)
    #model_preds_df = pd.read_csv(os.path.join(k_fold_folder_dir, "model_predictions.csv"), delimiter=";")
    df_fold_preds = pd.read_csv(os.path.join(k_fold_folder_dir, "all_lr_predictions_MDACC.csv"), delimiter=";")
    df_fold_preds[PATIENT_ID_COL_NAME] = df_fold_preds[PATIENT_ID_COL_NAME].astype(str).str.zfill(5)
    #df_fold_preds[PATIENT_ID_COL_NAME] = df_fold_preds[PATIENT_ID_COL_NAME].astype(str)
    df_test_preds = df_fold_preds[df_fold_preds["Mode"] == "test"]

    fold_test_set_preds_list.append(df_test_preds.copy())
    test_patientIDs_list = list(df_test_preds[PATIENT_ID_COL_NAME].values)

    if test_patientIDs_list_fold1 == None:
        test_patientIDs_list_fold1 = test_patientIDs_list
    else:
        assert test_patientIDs_list_fold1 == test_patientIDs_list, 'Patient IDs are not the same for all folds'





In [19]:
sum(all_test_preds_df['PatientID'] == 10785.0  )

NameError: name 'all_test_preds_df' is not defined

In [20]:
all_test_preds_df['PatientID']

NameError: name 'all_test_preds_df' is not defined

In [111]:
grouby_obj = all_test_preds_df.groupby(['PatientID', 'Mode'])

i = 0
for name, group in grouby_obj:
    i += 1
    if i == 5:
        print(name)
        print(group)
        print(group.shape)
        print("\n\n")
        break

('11481', 'test')
  PatientID  Mode  Aspiration_M06_pred  Aspiration_M06_true  \
4     11481  test             0.049933                    0   
4     11481  test             0.038182                    0   
4     11481  test             0.053404                    0   
4     11481  test             0.052463                    0   
4     11481  test             0.047881                    0   

   Dysphagia_M06_pred  Dysphagia_M06_true  Sticky_M06_pred  Sticky_M06_true  \
4            0.150111                   0         0.427222                1   
4            0.149725                   0         0.428135                1   
4            0.149815                   0         0.444377                1   
4            0.163571                   0         0.436119                1   
4            0.140043                   0         0.442849                1   

   Taste_M06_pred  Taste_M06_true  Xerostomia_M06_pred  Xerostomia_M06_true  
4        0.386917               0             0.54

In [21]:
# Concatenate all dataframes in the list
all_test_preds_df = pd.concat(fold_test_set_preds_list)

# Group by PatientID and Mode, then calculate the mean for each group
ensemble_test_preds_df = all_test_preds_df.groupby(['PatientID', 'Mode']).mean().reset_index()

print(ensemble_test_preds_df.head())

    PatientID  Mode  Aspiration_M06_pred  Aspiration_M06_true  \
0  1019236881  test             0.328669                  0.0   
1  1022096596  test             0.300029                  0.0   
2  1027008628  test             0.053384                  0.0   
3  1029341402  test             0.036912                  0.0   
4  1041973121  test             0.040628                  0.0   

   Dysphagia_M06_pred  Dysphagia_M06_true  Sticky_M06_pred  Sticky_M06_true  \
0            0.674296                 1.0         0.695460              1.0   
1            0.560782                 0.0         0.250126              0.0   
2            0.275473                 0.0         0.270694              0.0   
3            0.362215                 0.0         0.273383              0.0   
4            0.195399                 1.0         0.300791              0.0   

   Taste_M06_pred  Taste_M06_true  Xerostomia_M06_pred  Xerostomia_M06_true  
0        0.373084             1.0             0.812652  

In [23]:
config['general']['resultsCurrentDirectory'] = DL_trial_dir


LR_ensemble_preds_dir = os.path.join(DL_trial_dir, f"LR_ensemble_preds_{config['data']['source']}.csv")
LR_ensemble_preds_dir
ensemble_test_preds_df.to_csv(LR_ensemble_preds_dir, sep=";", index=False)
    #print("saved all predictions")s

pred_csv_dir = os.path.join(config['general']['resultsCurrentDirectory'], f"LR_ensemble_preds_{config['data']['source']}.csv")
total_evaluation_current_fold(config, sets = ['test'], pred_csv_dir = pred_csv_dir, lr=True)

get_visualizations(config, sets = ['test'], pred_csv_dir=pred_csv_dir, lr=True)

In [103]:
pred_csv_dir

'\\\\zkh\\appdata\\RTDicom\\Projectline_HNC_modelling\\Users\\Daniel MacRae\\1. MultiTox_HNC\\Feb_2025_results\\Trial_32\\LR_ensemble_preds_PRIMA.csv'

In [48]:
ensemble_test_preds_df

,PatientID,Mode,Aspiration_M06_pred,Aspiration_M06_true,Dysphagia_M06_pred,Dysphagia_M06_true,Sticky_M06_pred,Sticky_M06_true,Taste_M06_pred,Taste_M06_true,Xerostomia_M06_pred,Xerostomia_M06_true
0,10785.0,test,0.026837,0.0,0.070982,0.0,0.188626,1.0,0.115696,0.2,0.175571,0.2
1,11247.0,test,0.100089,0.0,0.282355,0.0,0.402892,1.0,0.375728,0.6,0.629594,1.0
2,11349.0,test,0.099504,0.0,0.274719,0.0,0.428055,0.4,0.314806,0.0,0.669073,1.0
3,11391.0,test,0.096674,0.0,0.187048,0.0,0.234368,0.0,0.180847,0.0,0.319015,0.2
4,11481.0,test,0.160097,0.0,0.362987,0.0,0.251392,0.2,0.321126,0.0,0.395197,0.2
...,...,...,...,...,...,...,...,...,...,...,...,...
270,99577.0,test,0.039453,0.0,0.138243,0.2,0.316936,0.4,0.333342,0.2,0.506276,0.2
271,99636.0,test,0.074999,0.0,0.269504,0.4,0.461436,0.8,0.287076,0.4,0.600917,0.8
272,99777.0,test,0.155588,0.0,0.376300,0.2,0.467985,0.4,0.264745,0.2,0.604753,0.4
273,99870.0,test,0.119633,0.4,0.350144,0.4,0.388443,0.0,0.367499,0.2,0.507442,0.2
